# Data Exploration and Enrichment
**Ethiopia Financial Inclusion Forecasting — Selam Analytics**

Objective: understand the starter dataset's schema, explore it, and enrich it with
additional sourced data useful for forecasting Access (account ownership) and
Usage (digital payment adoption).

All heavy logic lives in `src/data_loader.py` (loading + exploration) and
`src/enrichment.py` (adding new rows + logging). Both `add_record()` and
`append_log_entry()` are **idempotent** — re-running this notebook top to bottom
will not create duplicate rows or duplicate log entries.


In [24]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

from src.data_loader import (
    load_unified_data, load_reference_codes,
    get_observations, get_events, get_impact_links, get_targets,
    link_events_to_impacts,
    summarize_counts, get_temporal_range, get_indicator_coverage,
    get_events_catalog, review_impact_links, find_schema_violations,
)
from src.enrichment import add_record, append_log_entry

COLLECTED_BY = "Meron Sisay"          # <-- replace with your name
COLLECTION_DATE = "2026-07-17"        # <-- replace with today's date when you run this


## 1. Understand the Schema

Load the three raw files and confirm the unified-schema design:
- `observation` — measured values, `pillar` filled, `category` empty
- `event` — policies/launches/milestones, `category` filled, **`pillar` intentionally empty**
  (an event can affect multiple pillars, so pre-assigning one would bias the analysis)
- `impact_link` — modeled event → indicator relationships, joined via `parent_id`
- `target` — official policy goals


In [25]:
df = load_unified_data()
reference = load_reference_codes()

print(f"Loaded {len(df)} records across {df['record_type'].nunique()} record types")
df.head()


Loaded 57 records across 4 record types


,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,unit,observation_date,period_start,period_end,fiscal_year,gender,location,region,source_name,source_type,source_url,confidence,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes,parent_id
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,%,2014-12-31,NaT,NaT,2014,all,national,NaN,Global Findex 2014,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Baseline year,NaN,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,%,2017-12-31,NaT,NaT,2017,all,national,NaN,Global Findex 2017,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,all,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,male,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Gender disaggregated,NaN,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,%,2021-12-31,NaT,NaT,2021,female,national,NaN,Global Findex 2021,survey,https://www.worldbank.org/en/publication/globa...,high,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20 00:00:00,NaN,Gender disaggregated,NaN,NaN


In [26]:
reference.head(15)


,field,code,description,applies_to
0,record_type,observation,Actual measured value from a source,All
1,record_type,event,Policy launch market event or milestone,All
2,record_type,impact_link,Relationship between event and indicator (link...,All
3,record_type,target,Policy target or official goal,All
4,record_type,baseline,Starting point for comparison,All
5,record_type,forecast,Predicted future value,All
6,category,product_launch,New product or service introduced,event
7,category,market_entry,New competitor enters market,event
8,category,market_exit,Competitor leaves market,event
9,category,policy,Government strategy or regulatory framework,event


In [27]:
df.columns.tolist()


['record_id',
 'record_type',
 'category',
 'pillar',
 'indicator',
 'indicator_code',
 'indicator_direction',
 'value_numeric',
 'value_text',
 'value_type',
 'unit',
 'observation_date',
 'period_start',
 'period_end',
 'fiscal_year',
 'gender',
 'location',
 'region',
 'source_name',
 'source_type',
 'source_url',
 'confidence',
 'related_indicator',
 'relationship_type',
 'impact_direction',
 'impact_magnitude',
 'impact_estimate',
 'lag_months',
 'evidence_basis',
 'comparable_country',
 'collected_by',
 'collection_date',
 'original_text',
 'notes',
 'parent_id']

## 2. Explore the Data

Counts by `record_type`, `pillar`, `source_type`, `confidence`; temporal range;
indicator coverage; the event catalog; and a review of every impact_link.


In [28]:
for name, counts in summarize_counts(df).items():
    print(f"--- {name} ---")
    print(counts, "\n")


--- record_type ---
record_type
observation    30
impact_link    14
event          10
target          3
Name: count, dtype: int64 

--- pillar ---
pillar
ACCESS           20
USAGE            17
NaN              10
GENDER            6
AFFORDABILITY     4
Name: count, dtype: int64 

--- source_type ---
source_type
operator      15
NaN           14
survey        10
regulator      7
research       4
policy         3
calculated     2
news           2
Name: count, dtype: int64 

--- confidence ---
confidence
high      44
medium    13
Name: count, dtype: int64 



In [29]:
get_temporal_range(df)


min   2014-12-31
max   2025-12-31
Name: observation_date, dtype: datetime64[ns]

In [30]:
get_indicator_coverage(df)


,n_records,first_observed,last_observed
indicator_code,,,
ACC_OWNERSHIP,6,2014-12-31,2024-11-29
ACC_FAYDA,3,2024-08-15,2025-05-15
ACC_4G_COV,2,2023-06-30,2025-06-30
ACC_MM_ACCOUNT,2,2021-12-31,2024-11-29
GEN_GAP_ACC,2,2021-12-31,2024-11-29
USG_P2P_COUNT,2,2024-07-07,2025-07-07
USG_CROSSOVER,1,2025-07-07,2025-07-07
USG_TELEBIRR_USERS,1,2025-06-30,2025-06-30
USG_P2P_VALUE,1,2025-07-07,2025-07-07


In [31]:
get_events_catalog(df)


,record_id,indicator,category,observation_date,source_name
33,EVT_0001,Telebirr Launch,product_launch,2021-05-17,Ethio Telecom
41,EVT_0009,NFIS-II Strategy Launch,policy,2021-09-01,NBE
34,EVT_0002,Safaricom Ethiopia Commercial Launch,market_entry,2022-08-01,News
35,EVT_0003,M-Pesa Ethiopia Launch,product_launch,2023-08-01,Safaricom
36,EVT_0004,Fayda Digital ID Program Rollout,infrastructure,2024-01-01,NIDP
37,EVT_0005,Foreign Exchange Liberalization,policy,2024-07-29,NBE
38,EVT_0006,P2P Transaction Count Surpasses ATM,milestone,2024-10-01,EthSwitch
39,EVT_0007,M-Pesa EthSwitch Integration,partnership,2025-10-27,EthSwitch
42,EVT_0010,Safaricom Ethiopia Price Increase,pricing,2025-12-15,News
40,EVT_0008,EthioPay Instant Payment System Launch,infrastructure,2025-12-18,NBE/EthSwitch


In [32]:
review_impact_links(df)


,record_id_impact,indicator_event,category_event,pillar_impact,related_indicator_impact,impact_direction_impact,impact_magnitude_impact,lag_months_impact,evidence_basis_impact,confidence_impact
0,IMP_0001,Telebirr Launch,product_launch,ACCESS,ACC_OWNERSHIP,increase,high,12.0,literature,medium
1,IMP_0002,Telebirr Launch,product_launch,USAGE,USG_TELEBIRR_USERS,increase,high,3.0,empirical,high
2,IMP_0003,Telebirr Launch,product_launch,USAGE,USG_P2P_COUNT,increase,high,6.0,empirical,medium
3,IMP_0004,Safaricom Ethiopia Commercial Launch,market_entry,ACCESS,ACC_4G_COV,increase,medium,12.0,empirical,medium
4,IMP_0005,Safaricom Ethiopia Commercial Launch,market_entry,AFFORDABILITY,AFF_DATA_INCOME,decrease,medium,12.0,literature,medium
5,IMP_0006,M-Pesa Ethiopia Launch,product_launch,USAGE,USG_MPESA_USERS,increase,high,3.0,empirical,high
6,IMP_0007,M-Pesa Ethiopia Launch,product_launch,ACCESS,ACC_MM_ACCOUNT,increase,medium,6.0,theoretical,medium
7,IMP_0008,Fayda Digital ID Program Rollout,infrastructure,ACCESS,ACC_OWNERSHIP,increase,medium,24.0,literature,medium
8,IMP_0009,Fayda Digital ID Program Rollout,infrastructure,GENDER,GEN_GAP_ACC,decrease,medium,24.0,literature,medium
9,IMP_0010,Foreign Exchange Liberalization,policy,AFFORDABILITY,AFF_DATA_INCOME,increase,high,3.0,empirical,high


### Schema violations / corrections

`find_schema_violations` checks two things automatically:
1. Any `event` row that has a `pillar` filled in (shouldn't happen per the schema).
2. Any categorical value not present in `reference_codes.csv`.


In [33]:
violations = find_schema_violations(df, reference)
violations if not violations.empty else "None found by the automated check"


'None found by the automated check'

### Manual corrections found while exploring

The automated check only catches schema-rule violations, not factual/labeling errors.
Found by inspection:

**1. `REC_0006`'s `collection_date` column contains free text, not a date.**

```
record_id : REC_0006
collection_date : "Account ownership increased from 46% to 49%"
original_text    : "Survey Oct-Nov 2024"
```
That sentence belongs in `notes`/`original_text`, not `collection_date`. This is why
`load_unified_data()` deliberately does **not** auto-parse `collection_date` as a date.

**2. `REC_0004` / `REC_0005` (gender-disaggregated Account Ownership) are dated `2021-12-31`,**
**but the values match reporting on the 2024 Findex round, not 2021.**

> "There was also a marked gender gap, with 56% of men owning accounts compared to only 36% of women."
> — DFS Ethiopia Hub / Shega, on the Global Findex 2025 release (survey fielded Oct 15–Nov 29, 2024)
https://digitalfinance.shega.co/insights/articles/findex-2025-and-ethiopia-s-digital-financial-leap-momentum-without-maturity

The Findex 2024 survey window (`2024-10-15` to `2024-11-29`) matches `period_start`/`period_end`
already present elsewhere in this dataset for the 2024 round. Corrected below.

** Important caveat found later, while enriching (section 3):** a second, independently
reported source gives a *different* 2024 gender split — **57% men / 42% women** — not 56%/36%.
Both numbers are attributed to "Global Findex 2025" by otherwise-reputable outlets. This is a
genuine conflict between secondary sources, not something this notebook can resolve on its own.
**Recommendation: verify against the primary World Bank Findex microdata
(https://microdata.worldbank.org/index.php/catalog/7901) before using this figure in Task 3/4
modeling.** Logged as a flagged data-quality issue, not silently resolved.


In [34]:
# Inspect the rows in question before deciding whether to correct them
df[df["record_id"].isin(["REC_0004", "REC_0005", "REC_0006"])][
    ["record_id", "indicator", "value_numeric", "observation_date", "collection_date", "original_text"]
]


,record_id,indicator,value_numeric,observation_date,collection_date,original_text
3,REC_0004,Account Ownership Rate,56.0,2021-12-31,NaN,Gender disaggregated
4,REC_0005,Account Ownership Rate,36.0,2021-12-31,NaN,Gender disaggregated
5,REC_0006,Account Ownership Rate,49.0,2024-11-29,Account ownership increased from 46% to 49%,Survey Oct-Nov 2024


In [35]:
# Apply the date correction for REC_0004 / REC_0005 (see reasoning above)
df.loc[df["record_id"].isin(["REC_0004", "REC_0005"]), "observation_date"] = pd.Timestamp("2024-11-29")

append_log_entry(
    record_id="REC_0004, REC_0005",
    record_type="observation (correction)",
    description="Corrected observation_date from 2021-12-31 to 2024-11-29",
    source_url="https://digitalfinance.shega.co/insights/articles/findex-2025-and-ethiopia-s-digital-financial-leap-momentum-without-maturity",
    original_text="There was also a marked gender gap, with 56% of men owning accounts compared to only 36% of women.",
    confidence="medium",
    collected_by=COLLECTED_BY,
    collection_date=COLLECTION_DATE,
    notes=("The 56%/36% gender split matches the 2024 Findex round (survey window "
           "2024-10-15 to 2024-11-29), not the 2021 round these rows were originally "
           "dated to. NOTE: a conflicting 57%/42% figure for the same 2024 round was found "
           "later (see section 3) -- flagged for verification, not resolved here."),
)

append_log_entry(
    record_id="REC_0004, REC_0005 (flag)",
    record_type="data_quality_flag",
    description="Conflicting secondary sources for 2024 gender gap: 56/36 vs 57/42",
    source_url="https://birrmetrics.com/49-of-ethiopians-are-banked-as-findex-2025-highlights-the-next-inclusion-challenge/",
    original_text="While 57 percent of men in Ethiopia report having an account, only 42 percent of women do.",
    confidence="low",
    collected_by=COLLECTED_BY,
    collection_date=COLLECTION_DATE,
    notes=("Birr Metrics (citing the World Bank Findex 2025 release directly) reports 57%/42%, "
           "not 56%/36% as reported by Shega/DFS Ethiopia Hub. Both cite the same underlying "
           "survey. Not resolved in this dataset -- flagging for verification against primary "
           "Findex microdata before relying on either number in forecasting."),
)
print("Logged correction + conflict flag for REC_0004 / REC_0005")


Logged correction + conflict flag for REC_0004 / REC_0005


## 3. Enrich the Dataset

Two batches of new records, both sourced from GSMA and Ethiopian digital-finance coverage
found via web search, chosen to close gaps the project brief explicitly calls out.


### Batch 1 — Registered-vs-active gap and agent network (Access)

Findex counts *active, self-reported* mobile money use (`ACC_MM_ACCOUNT` = 9.45% in 2024).
GSMA's operator-reported figure of 90M+ *registered* mobile accounts is a completely
different, much larger number for the same underlying system — the gap between these two
numbers is exactly the "registered vs. active" phenomenon Task 2 is supposed to investigate.
Agent density is a leading indicator for Access (GSMA's own research finds distance to
an agent is the top reason people don't open a mobile money account).


In [36]:
new_records_batch1 = [
    {
        "record_type": "observation", "pillar": "ACCESS",
        "indicator": "Registered Mobile Money Accounts (cumulative)",
        "indicator_code": "ACC_MM_REGISTERED",
        "value_numeric": 90_000_000, "value_type": "count", "unit": "accounts",
        "observation_date": "2024-10-24",
        "source_name": "GSMA", "source_type": "operator",
        "source_url": "https://www.gsma.com/newsroom/press-release/ethiopias-digital-economy-to-contribute-etb-1-3-trillion-to-gdp-by-2028-unlocking-jobs-and-growth-through-telecom-and-policy-reforms/",
        "confidence": "high",
        "original_text": "Ethiopia has already made significant strides, with over 90 million registered mobile accounts by 2024.",
        "notes": ("Registered accounts vastly exceed Findex's active-use ACC_MM_ACCOUNT (9.45%). "
                  "Central to explaining why account ownership stagnated despite mass account opening."),
    },
    {
        "record_type": "observation", "pillar": "ACCESS",
        "indicator": "Mobile Money Agent Count", "indicator_code": "ACC_AGENT_COUNT",
        "value_numeric": 200_000, "value_type": "count", "unit": "agents",
        "observation_date": "2022-09-30",
        "source_name": "National Bank of Ethiopia (via GSMA)", "source_type": "regulator",
        "source_url": "https://www.gsma.com/mobilefordevelopment/blog/mobile-money-in-ethiopia-what-we-learnt-from-our-expert-roundtable/",
        "confidence": "medium",
        "original_text": "mobile money agents grew by 200% in the year to September 2022 to over 200,000.",
        "notes": "Early agent-network growth point, ~16 months after Telebirr's launch.",
    },
    {
        "record_type": "observation", "pillar": "ACCESS",
        "indicator": "Mobile Money Agent Count", "indicator_code": "ACC_AGENT_COUNT",
        "value_numeric": 216_000, "value_type": "count", "unit": "agents",
        "observation_date": "2024-06-30",
        "source_name": "Shega / DFS Ethiopia Hub", "source_type": "news",
        "source_url": "https://shega.co/news/the-rise-of-mobile-money-in-ethiopia-without-the-agents",
        "confidence": "medium",
        "original_text": "...had around 216,000 agents as of June 2024.",
        "notes": ("Growth from 200k (2022) to 216k (2024) is much slower than account growth -- "
                  "candidate explanation for the Access slowdown."),
    },
]

for record in new_records_batch1:
    df, added = add_record(df, record)
    if added:
        logged = append_log_entry(
            record_id=df.iloc[-1]["record_id"], record_type=record["record_type"],
            description=f"{record['indicator']} ({record['observation_date']})",
            source_url=record["source_url"], original_text=record["original_text"],
            confidence=record["confidence"], collected_by=COLLECTED_BY,
            collection_date=COLLECTION_DATE, notes=record["notes"],
        )
        print(f"Added {df.iloc[-1]['record_id']}: {record['indicator']}")
    else:
        print(f"Skipped (already present): {record['indicator']} ({record['observation_date']})")


Added REC_0034: Registered Mobile Money Accounts (cumulative)
Added REC_0035: Mobile Money Agent Count
Added REC_0036: Mobile Money Agent Count


In [38]:
# New impact_link: Telebirr launch -> agent network growth (not previously captured)
df, added = add_record(df, {
    "record_type": "impact_link", "parent_id": "EVT_0001", "pillar": "ACCESS",
    "related_indicator": "ACC_AGENT_COUNT", "relationship_type": "direct",
    "impact_direction": "increase", "impact_magnitude": "high", "lag_months": 16,
    "evidence_basis": "empirical", "confidence": "medium",
    "source_name": "GSMA",
    "source_url": "https://www.gsma.com/mobilefordevelopment/blog/mobile-money-in-ethiopia-what-we-learnt-from-our-expert-roundtable/",
    "original_text": "mobile money agents grew by 200% in the year to September 2022 to over 200,000.",
})
if added:
    append_log_entry(
        record_id=df.iloc[-1]["record_id"], record_type="impact_link",
        description="Telebirr launch -> ACC_AGENT_COUNT",
        source_url="https://www.gsma.com/mobilefordevelopment/blog/mobile-money-in-ethiopia-what-we-learnt-from-our-expert-roundtable/",
        original_text="mobile money agents grew by 200% in the year to September 2022 to over 200,000.",
        confidence="medium", collected_by=COLLECTED_BY, collection_date=COLLECTION_DATE,
        notes="Connects the Telebirr launch event to observed agent-network growth ~16 months later.",
    )
    print(f"Added {df.iloc[-1]['record_id']}")
else:
    print("Skipped (already present): Telebirr -> ACC_AGENT_COUNT")


Skipped (already present): Telebirr -> ACC_AGENT_COUNT


### Batch 2 — Digital Payment Usage, disaggregations, and enablers

**Critical gap found in section 2**: `USG_DIGITAL_PAYMENT` — literally one of the two headline
indicators this whole project forecasts — had **zero observations** in the starter dataset.
The observations below close that gap, plus add income/gender disaggregation and infrastructure
enablers (smartphone/phone ownership) called out in the project's enrichment guide.

** Digital payment figure conflict, flagged rather than silently resolved:**
the original challenge brief cites "~35%" for 2024 digital payment adoption. Direct research
into the Findex 2025 release instead finds **21%** (Birr Metrics, citing World Bank Findex
data directly, with numbers that internally cross-check against several other known figures
in this dataset). A separate source implies ~23% for 2021 and "~1pp change" between rounds,
which is *consistent with* 21-23%, not with 35%. **This notebook uses 21%/23% (the better-
corroborated figures) and logs the conflict explicitly rather than guessing which is right.**


In [39]:
new_records_batch2 = [
    {
        "record_type": "observation", "pillar": "USAGE",
        "indicator": "Digital Payment Adoption Rate", "indicator_code": "USG_DIGITAL_PAYMENT",
        "value_numeric": 23, "value_type": "percentage", "unit": "%",
        "observation_date": "2021-12-31",
        "source_name": "Shega / DFS Ethiopia Hub", "source_type": "survey",
        "source_url": "https://shega.co/news/findex-2025-and-ethiopias-digital-financial-leap-momentum-without-maturity",
        "confidence": "medium",
        "original_text": "Mobile money accounts were at 4.7% and digital payments were used by fewer than one in four adults.",
        "notes": ("Approximate -- source states '<25%', not an exact figure. Fills a previously "
                  "empty indicator that is one of the two headline forecast targets."),
    },
    {
        "record_type": "observation", "pillar": "USAGE",
        "indicator": "Digital Payment Adoption Rate", "indicator_code": "USG_DIGITAL_PAYMENT",
        "value_numeric": 21, "value_type": "percentage", "unit": "%",
        "observation_date": "2024-11-29",
        "source_name": "Birr Metrics (citing World Bank Global Findex 2025)", "source_type": "survey",
        "source_url": "https://birrmetrics.com/49-of-ethiopians-are-banked-as-findex-2025-highlights-the-next-inclusion-challenge/",
        "confidence": "high",
        "original_text": "36 percent saved in account with 21 percent using digital payments.",
        "notes": ("CONFLICTS with project brief's ~35% figure -- see caveat above. This figure "
                  "chosen because it cross-checks against other independently known 2024 figures "
                  "(49% account ownership, 22%->49% trend) reported correctly in the same article."),
    },
    {
        "record_type": "observation", "pillar": "ACCESS",
        "indicator": "Smartphone Penetration", "indicator_code": "ACC_SMARTPHONE_PEN",
        "value_numeric": 16, "value_type": "percentage", "unit": "%",
        "observation_date": "2024-11-29",
        "source_name": "Birr Metrics (citing World Bank Global Findex 2025)", "source_type": "survey",
        "source_url": "https://birrmetrics.com/49-of-ethiopians-are-banked-as-findex-2025-highlights-the-next-inclusion-challenge/",
        "confidence": "high",
        "original_text": "Smartphone penetration stands at only 16 percent, with most users relying on feature or basic phones.",
        "notes": "Enabler/proxy variable (Sheet C of the enrichment guide) -- low smartphone penetration caps digital Usage growth even as Access rises.",
    },
    {
        "record_type": "observation", "pillar": "ACCESS",
        "indicator": "Mobile Phone Ownership", "indicator_code": "ACC_PHONE_OWNERSHIP",
        "value_numeric": 41, "value_type": "percentage", "unit": "%",
        "observation_date": "2024-11-29",
        "source_name": "Birr Metrics (citing World Bank Global Findex 2025)", "source_type": "survey",
        "source_url": "https://birrmetrics.com/49-of-ethiopians-are-banked-as-findex-2025-highlights-the-next-inclusion-challenge/",
        "confidence": "high",
        "original_text": "only 41 percent of adults own a mobile phone, with stark gender differences -- 50 percent of men compared to just 33 percent of women.",
        "notes": "Basic phone ownership is a prerequisite enabler for both Access (mobile money) and Usage (digital payments).",
    },
    {
        "record_type": "observation", "pillar": "ACCESS",
        "indicator": "Account Ownership, wealthiest 60%", "indicator_code": "ACC_OWNERSHIP_TOP60",
        "value_numeric": 53, "value_type": "percentage", "unit": "%",
        "observation_date": "2024-11-29",
        "source_name": "Birr Metrics (citing World Bank Global Findex 2025)", "source_type": "survey",
        "source_url": "https://birrmetrics.com/49-of-ethiopians-are-banked-as-findex-2025-highlights-the-next-inclusion-challenge/",
        "confidence": "high",
        "original_text": "account ownership among the wealthiest 60 percent stands at 53 percent, compared with 43 percent among the poorest 40 percent.",
        "notes": "Income disaggregation of Access, not previously captured -- Sheet C enabler/equity dimension.",
    },
    {
        "record_type": "observation", "pillar": "ACCESS",
        "indicator": "Account Ownership, poorest 40%", "indicator_code": "ACC_OWNERSHIP_BOTTOM40",
        "value_numeric": 43, "value_type": "percentage", "unit": "%",
        "observation_date": "2024-11-29",
        "source_name": "Birr Metrics (citing World Bank Global Findex 2025)", "source_type": "survey",
        "source_url": "https://birrmetrics.com/49-of-ethiopians-are-banked-as-findex-2025-highlights-the-next-inclusion-challenge/",
        "confidence": "high",
        "original_text": "compared with 43 percent among the poorest 40 percent.",
        "notes": "Paired with ACC_OWNERSHIP_TOP60 -- a 10pp income gap in Access.",
    },
    {
        "record_type": "observation", "pillar": "USAGE",
        "indicator": "Digital Payment Usage, wealthiest 60%", "indicator_code": "USG_DIGITAL_PAYMENT_TOP60",
        "value_numeric": 26, "value_type": "percentage", "unit": "%",
        "observation_date": "2024-11-29",
        "source_name": "Birr Metrics (citing World Bank Global Findex 2025)", "source_type": "survey",
        "source_url": "https://birrmetrics.com/49-of-ethiopians-are-banked-as-findex-2025-highlights-the-next-inclusion-challenge/",
        "confidence": "high",
        "original_text": "26 percent of the wealthiest 60 percent reported digital transactions, compared to 16 percent of the poorest.",
        "notes": "Income disaggregation of Usage -- larger relative gap than Access (10pp income gap in Access vs. 10pp here on a smaller base).",
    },
    {
        "record_type": "observation", "pillar": "USAGE",
        "indicator": "Digital Payment Usage, poorest 40%", "indicator_code": "USG_DIGITAL_PAYMENT_BOTTOM40",
        "value_numeric": 16, "value_type": "percentage", "unit": "%",
        "observation_date": "2024-11-29",
        "source_name": "Birr Metrics (citing World Bank Global Findex 2025)", "source_type": "survey",
        "source_url": "https://birrmetrics.com/49-of-ethiopians-are-banked-as-findex-2025-highlights-the-next-inclusion-challenge/",
        "confidence": "high",
        "original_text": "compared to 16 percent of the poorest.",
        "notes": "Paired with USG_DIGITAL_PAYMENT_TOP60.",
    },
    {
        "record_type": "observation", "pillar": "GENDER",
        "indicator": "Digital Payment Usage Gender Gap", "indicator_code": "GEN_DIGITAL_PAYMENT_GAP",
        "value_numeric": 13, "value_type": "percentage_points", "unit": "pp",
        "observation_date": "2024-11-29",
        "source_name": "Birr Metrics (citing World Bank Global Findex 2025)", "source_type": "survey",
        "source_url": "https://birrmetrics.com/49-of-ethiopians-are-banked-as-findex-2025-highlights-the-next-inclusion-challenge/",
        "confidence": "high",
        "original_text": "Only 26 percent of men and 13 percent of women used digital payments in the past year.",
        "notes": "Gender gap in Usage (13pp) is proportionally much larger than the ~15-20pp gap in Access, given the smaller overall base -- worth testing in Task 2.",
    },
]

for record in new_records_batch2:
    df, added = add_record(df, record)
    if added:
        append_log_entry(
            record_id=df.iloc[-1]["record_id"], record_type=record["record_type"],
            description=f"{record['indicator']} ({record['observation_date']})",
            source_url=record["source_url"], original_text=record["original_text"],
            confidence=record["confidence"], collected_by=COLLECTED_BY,
            collection_date=COLLECTION_DATE, notes=record["notes"],
        )
        print(f"Added {df.iloc[-1]['record_id']}: {record['indicator']}")
    else:
        print(f"Skipped (already present): {record['indicator']} ({record['observation_date']})")


Added REC_0037: Digital Payment Adoption Rate
Added REC_0038: Digital Payment Adoption Rate
Added REC_0039: Smartphone Penetration
Added REC_0040: Mobile Phone Ownership
Added REC_0041: Account Ownership, wealthiest 60%
Added REC_0042: Account Ownership, poorest 40%
Added REC_0043: Digital Payment Usage, wealthiest 60%
Added REC_0044: Digital Payment Usage, poorest 40%
Added REC_0045: Digital Payment Usage Gender Gap


### Batch 3 — New event 

Batches 1-2 only added `observation` rows and one `impact_link` --  This closes that gap with a
real regulatory event not in the starter catalog: NBE's October 2023 revision of the Payment
Instrument Issuer Directive for mobile money providers.


In [40]:
df, added = add_record(df, {
    "record_type": "event",
    "category": "regulation",
    # pillar deliberately left out -- this directive plausibly affects both
    # ACCESS (banks can now spin up mobile money subsidiaries) and USAGE
    # (higher transaction limits enable more/larger digital payments), so
    # per the schema's core design principle it should NOT be pre-assigned.
    "indicator": "NBE Revised Payment Instrument Issuer Directive (mobile money limits)",
    "observation_date": "2023-10-09",
    "source_name": "National Bank of Ethiopia / Telecom Review Africa",
})

if added:
    new_event_id = df.iloc[-1]["record_id"]
    append_log_entry(
        record_id=new_event_id, record_type="event",
        description="NBE Revised Payment Instrument Issuer Directive, Oct 2023",
        source_url="https://www.telecomreviewafrica.com/articles/general-news/3842-nbe-enhances-directive-for-mobile-money-services/",
        original_text=("the daily e-account balance limit has been increased from 30,000 birr "
                        "($539) to 75,000 birr ($1,347), and a new daily global transaction limit "
                        "of 150,000 birr ($2,695) has been implemented."),
        confidence="high", collected_by=COLLECTED_BY, collection_date=COLLECTION_DATE,
        notes=("Regulatory event missing from the starter catalog. Category='regulation', "
               "pillar deliberately left empty since it plausibly affects both ACCESS "
               "(banks can launch mobile money subsidiaries) and USAGE (higher transaction "
               "limits enable more/larger digital payments)."),
    )
    print(f"Added event {new_event_id}")
else:
    print("Skipped (already present): NBE Revised Payment Instrument Issuer Directive")
    # re-derive the id for the impact_link cell below, in case this cell is
    # re-run on its own after the event already exists
    existing = df[(df["record_type"] == "event") &
                  (df["indicator"] == "NBE Revised Payment Instrument Issuer Directive (mobile money limits)")]
    new_event_id = existing.iloc[0]["record_id"]


Added event EVT_0011


In [41]:
# Two impact_links for the new event -- theoretical/literature-based since
# there's no clean Ethiopia-specific before/after data isolating just this
# directive's effect (it overlaps in time with several other 2023 events).
new_impact_links = [
    {
        "record_type": "impact_link", "parent_id": new_event_id, "pillar": "USAGE",
        "related_indicator": "USG_P2P_VALUE", "relationship_type": "direct",
        "impact_direction": "increase", "impact_magnitude": "medium", "lag_months": 6,
        "evidence_basis": "theoretical", "confidence": "medium",
        "source_name": "National Bank of Ethiopia / Telecom Review Africa",
        "source_url": "https://www.telecomreviewafrica.com/articles/general-news/3842-nbe-enhances-directive-for-mobile-money-services/",
        "original_text": "the daily e-account balance limit has been increased from 30,000 birr to 75,000 birr...",
        "notes": "Higher balance/transaction ceilings directly enable larger-value P2P transfers.",
    },
    {
        "record_type": "impact_link", "parent_id": new_event_id, "pillar": "ACCESS",
        "related_indicator": "ACC_MM_ACCOUNT", "relationship_type": "indirect",
        "impact_direction": "increase", "impact_magnitude": "low", "lag_months": 12,
        "evidence_basis": "theoretical", "confidence": "low",
        "source_name": "National Bank of Ethiopia / Telecom Review Africa",
        "source_url": "https://www.telecomreviewafrica.com/articles/general-news/3842-nbe-enhances-directive-for-mobile-money-services/",
        "original_text": "the directive allows banks to establish subsidiaries specializing in the provision of mobile money services.",
        "notes": "Allowing banks to launch mobile-money subsidiaries is a plausible but indirect, slow-moving driver of account growth -- low confidence, no direct evidence yet.",
    },
]

for record in new_impact_links:
    df, added = add_record(df, record)
    if added:
        append_log_entry(
            record_id=df.iloc[-1]["record_id"], record_type="impact_link",
            description=f"NBE directive -> {record['related_indicator']}",
            source_url=record["source_url"], original_text=record["original_text"],
            confidence=record["confidence"], collected_by=COLLECTED_BY,
            collection_date=COLLECTION_DATE, notes=record["notes"],
        )
        print(f"Added {df.iloc[-1]['record_id']}")
    else:
        print(f"Skipped (already present): NBE directive -> {record['related_indicator']}")


Added IMP_0016
Added IMP_0017


## 4. Re-run the schema report on the enriched dataset

Sanity check: record counts should reflect all additions, no new schema violations,
and `USG_DIGITAL_PAYMENT` should now actually exist.


In [42]:
for name, counts in summarize_counts(df).items():
    print(f"--- {name} ---")
    print(counts, "\n")


--- record_type ---
record_type
observation    42
impact_link    17
event          11
target          3
Name: count, dtype: int64 

--- pillar ---
pillar
ACCESS           29
USAGE            22
NaN              11
GENDER            7
AFFORDABILITY     4
Name: count, dtype: int64 

--- source_type ---
source_type
survey        19
NaN           18
operator      16
regulator      8
research       4
policy         3
news           3
calculated     2
Name: count, dtype: int64 

--- confidence ---
confidence
high      53
medium    18
NaN        1
low        1
Name: count, dtype: int64 



In [43]:
get_indicator_coverage(df)


,n_records,first_observed,last_observed
indicator_code,,,
ACC_OWNERSHIP,6,2014-12-31,2024-11-29
ACC_FAYDA,3,2024-08-15,2025-05-15
ACC_4G_COV,2,2023-06-30,2025-06-30
ACC_MM_ACCOUNT,2,2021-12-31,2024-11-29
ACC_AGENT_COUNT,2,2022-09-30,2024-06-30
USG_P2P_COUNT,2,2024-07-07,2025-07-07
GEN_GAP_ACC,2,2021-12-31,2024-11-29
USG_DIGITAL_PAYMENT,2,2021-12-31,2024-11-29
USG_DIGITAL_PAYMENT_BOTTOM40,1,2024-11-29,2024-11-29


In [46]:
violations = find_schema_violations(df, reference)
violations if not violations.empty else "None found by the automated check"

'None found by the automated check'

## 5. Save the enriched dataset

Overwrite `data/raw/ethiopia_fi_unified_data.csv` with the corrected + enriched version.
`data_enrichment_log.md` (project root) has one entry per change made above.


In [47]:
out_path = Path.cwd().parent / "data" / "raw" / "ethiopia_fi_unified_data.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df)} records to {out_path}")


Saved 73 records to c:\Users\sisay\OneDrive\Documents\kaim\ethiopia-fi-forecast\data\raw\ethiopia_fi_unified_data.csv


## Key findings from this notebook

1. **Schema is clean at the rule level** — no events with a pre-assigned pillar, no invalid
   categorical codes.
2. **One correction + one unresolved conflict flag**: `REC_0004`/`REC_0005`'s date was corrected
   to 2024, but a *second* conflicting number for the same 2024 gender split (57/42 vs 56/36)
   surfaced during enrichment and is logged as needing verification against primary Findex data
   — not silently picked.
3. **`USG_DIGITAL_PAYMENT` — one of the two headline forecast targets — had zero data points
   before this notebook.** Two are added now (2021, 2024), with the 2024 figure itself flagged
   as conflicting with the project brief's stated ~35%.
4. **Registered-vs-active gap quantified**: 90M+ registered mobile accounts vs. 9.45%
   Findex-reported active use — a strong candidate explanation for Task 2's central puzzle.
5. **New income and gender disaggregations** for both Access and Usage, plus two infrastructure
   enablers (smartphone penetration, phone ownership) — useful leading indicators for Task 3/4.
6. All additions are logged in `data_enrichment_log.md` with source, exact quote, confidence,
   and rationale.

## Known limitations / what Task 2 should watch for

- Multiple secondary sources report *different* numbers for the same Findex 2024 release
  (gender gap, digital payment rate). This dataset is now internally consistent, but the
  underlying secondary sources are not consistent with each other -- worth a primary-source
  check before final modeling.
- Several new indicators (income quintile splits, gender-gap-in-usage) have only a single
  2024 data point -- no trend yet, so they're useful for cross-sectional analysis (Task 2)
  but not yet for time-series forecasting (Task 4) on their own.
